# Lab 25-10 · Qualification Equivalence and FMEDA Metrics

**Predict first**, then run. The handout is [README.md](README.md).

In [1]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
from scipy import optimize
import qual as q

## Part 1 · Coverage requirement

*Predict* the lockstep coverage needed for a 99% SPFM.

In [2]:
dc = optimize.brentq(lambda d: q.fmeda(200, 80, d, 20, 0.9)["spfm"] - 0.99, 0.5, 0.9999)
print(f"coverage for 99% SPFM: {100*dc:.1f} %")
dcs = np.linspace(0.9, 0.999, 60)
covs = np.linspace(0, 0.99, 60)
fig, ax = plt.subplots(1, 2, figsize=(9, 3.4))
ax[0].plot(100 * dcs, [100 * q.fmeda(200, 80, d, 20, 0.9)["spfm"] for d in dcs])
ax[0].axhline(99, ls=":", color="k"); ax[0].set_xlabel("lockstep coverage (%)"); ax[0].set_ylabel("SPFM (%)")
ax[1].plot(100 * covs, [100 * q.fmeda(200, 80, 0.99, 20, c)["lfm"] for c in covs])
ax[1].axhline(90, ls=":", color="k"); ax[1].set_xlabel("self-test coverage (%)"); ax[1].set_ylabel("LFM (%)")
plt.show()

coverage for 99% SPFM: 98.2 %


## Part 2 · Activation energy

*Predict* the stress hours at 150 °C with 0.5 eV.

In [3]:
prof = [(-40, 0.05), (70, 0.65), (110, 0.20), (125, 0.10)]
print(f"0.5 eV: {q.profile_stress_hours(prof, 20000, 0.5, 150):.0f} h; 0.7 eV: {q.profile_stress_hours(prof, 20000, 0.7, 150):.0f} h")
ea = np.linspace(0.4, 1.0, 40)
plt.figure()
for ts in (125, 150):
    plt.semilogy(ea, [q.profile_stress_hours(prof, 20000, e, ts) for e in ea], label=f"{ts} C")
plt.axhline(1000, ls=":", color="k")
plt.xlabel("activation energy (eV)"); plt.ylabel("stress hours to cover profile"); plt.legend(); plt.grid(True)
plt.show()

0.5 eV: 2333 h; 0.7 eV: 1286 h


## Part 3 · Latent faults by Monte Carlo

*Predict* the ratio of the simulated dual-point rate to λ_L λ_D T/2.

In [4]:
sim = q.simulate_dual(2e-6, 2e-5, 1000, 15000)
approx = 2e-6 * 2e-5 * 1000 / 2
print(f"simulated {sim:.3g} /h, approximation {approx:.3g} /h, ratio {sim/approx:.3f}")
tests = [100, 300, 1000, 3000, 7500, 15000]
plt.figure()
plt.loglog(tests, [q.simulate_dual(2e-6, 2e-5, t, 15000, 300_000) for t in tests], "o-", label="simulated")
plt.loglog(tests, [2e-6 * 2e-5 * t / 2 for t in tests], "--", label="lambda_L lambda_D T/2")
plt.xlabel("test interval (h)"); plt.ylabel("dual-point rate (1/h)"); plt.legend(); plt.grid(True)
plt.show()

simulated 1.96e-08 /h, approximation 2e-08 /h, ratio 0.982


## Part 4 · Vehicles

*Predict* the fraction of vehicles affected with 20,000 components at 0.1 ppm.

In [5]:
print(f"{100*q.vehicle_defect_fraction(20000, 1e-7):.3f} %")
p = np.logspace(-10, -5, 50)
plt.figure()
plt.loglog(p * 1e6, [q.vehicle_defect_fraction(20000, x) for x in p])
plt.xlabel("component defect rate (ppm)"); plt.ylabel("fraction of vehicles affected"); plt.grid(True)
plt.show()

0.200 %
